In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime
# from google.colab import files

# =========================
# BIST HİSSE LİSTESİ
# =========================
!pip install tradingview-screener -q

from tradingview_screener import Query, col

def get_bist_symbols_from_tradingview():
    try:
        _, df_tv = (
            Query()
            .select("name", "exchange", "type")
            .where(
                col("exchange") == "BIST",
                col("type") == "stock"
            )
            .limit(1000)
            .get_scanner_data()
        )

        symbols = df_tv["name"].dropna().unique().tolist()
        symbols = [s.replace("BIST:", "").strip() for s in symbols]
        symbols = sorted(list(set(symbols)))



    except Exception as e:
        print("TradingView liste çekme hatası:", e)
        return []

symbols = """
THYAO ASELS KCHOL SAHOL EREGL SISE TUPRS AKBNK GARAN YKBNK ISCTR TOASO FROTO ENKAI PETKM KOZAL ASTOR SASA HEKTS GOKNR
BIMAS CCOLA MGROS ULKER AEFES DOAS OTKAR ARCLK VESTL VESBE ALARK CIMSA OYAKC KONTR SMRTG EUPWR ODAS ZOREN EKGYO ENERY
AKSEN ALFAS GESAN CWENE YEOTK TTRAK KARSN TMSN AGHOL DOHOL TKFEN KORDS BRISA AKSA ALKIM SOKM TAVHL PGSUS DOCO TCELL
TTKOM LOGO MIATK ARDYZ NETAS KAREL QUAGR BIENY KLSER BOBET BUCIM ISGYO TRGYO VAKBN HALKB TSKB ALBRK SKBNK ECILC DEVA
MPARK GENIL SELEC ENJSA AYGAZ AKCNS ANSGR TURSG ULUUN KONYA MAGEN INVES KMPUR POLHO GLYHO CANTE GWIND ISMEN BRSAN
ASUZU ANHYT KCAER VAKKO YYLGD KLRHO KZBGY AKFGY INDES NUHCM SARKY AKFYE ISDMR GOZDE KRDMD KRDMA KRDMB ALCTL ALKA ALMAD
ANELE ANGEN ARSAN ATATP AYDEM BAGFS BASGZ BERA BFREN BIGCH BIZIM BRYAT BTCIM BURCE CEMTS CLEBI CRFSA DESA DESPC DGATE
DGNMO DITAS DNISI DOKTA DURDO DZGYO EDATA EGEEN EGGUB EGPRO EKOS ELITE ERBOS ERCB ESCAR ESCOM FORMT FRIGO GEDIK GEDZA
GENTS GEREL GLBMD GOLTS GOODY GRSEL GUBRF HATEK HDFGS HUBVC IHAAS IHEVA IHGZT IHLAS IHLGM IHYP IZMDC JANTS KAPLM KARTN
KATMR KERVT KFEIN KIMMR KLKIM KNFRT KRONT KRVGD KUTPO LKMNH MAALT MACKO MAVI MERKO METRO METUR MNDRS NIBAS NTGAZ
NTHOL ORGE OSTIM OYAYO PAPIL PENGD PINSU PKART PRKAB PRKME PSDTC RALYH RAYSG RYSAS SAFKR SAMAT SAYAS SEKFK SEKUR SILVR
SMART SNICA SOKE SUNTK TATGD TBORG TEKTU TGSAS TKNSA TLMAN TMPOL TSPOR TUCLK TURGG UFUK ULAS USAK UZERB VAKFN
VERUS VKGYO YATAS YAYLA YGYO YKSLN YUNSA ZEDUR
""".split()

symbols = sorted(list(set(symbols)))

print(f"Taranacak hisse sayısı: {len(symbols)}")

if len(symbols) == 0:
    print("Liste çekilemedi, eski liste kullanılacak.")

symbols = sorted(list(set(symbols)))

# =========================
# VERİ ÇEKME
# =========================
def download_data(symbol):
    ticker = symbol + ".IS"

    df = yf.download(
        ticker,
        period="2y",
        interval="1d",
        progress=False,
        auto_adjust=True,
        group_by="column"
    )

    if df is None or df.empty:
        return None

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    required = ["Open", "High", "Low", "Close", "Volume"]

    if not all(col in df.columns for col in required):
        return None

    df = df[required].copy()
    df = df.apply(pd.to_numeric, errors="coerce")
    df = df.dropna()

    if len(df) < 60:
        return None

    return df


def resample_ohlcv(df, rule):
    out = pd.concat({
        "Open": df["Open"].resample(rule).first(),
        "High": df["High"].resample(rule).max(),
        "Low": df["Low"].resample(rule).min(),
        "Close": df["Close"].resample(rule).last(),
        "Volume": df["Volume"].resample(rule).sum()
    }, axis=1)

    out = out.dropna()
    return out

# =========================
# İNDİKATÖRLER
# =========================
def rsi(series, length=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.rolling(length).mean()
    avg_loss = loss.rolling(length).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def mfi(df, length=14):
    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    mf = tp * df["Volume"]
    direction = tp.diff()

    pos = mf.where(direction > 0, 0).rolling(length).sum()
    neg = mf.where(direction < 0, 0).rolling(length).sum()

    return 100 - (100 / (1 + pos / neg.replace(0, np.nan)))

# =========================
# SİNYAL MOTORU
# =========================
def signal_engine(df):
    if df is None or len(df) < 30:
        return "YETERSİZ", 0

    close = df["Close"]
    high = df["High"]
    low = df["Low"]
    open_ = df["Open"]
    volume = df["Volume"]

    ema20 = close.ewm(span=20, adjust=False).mean()
    ema50 = close.ewm(span=50, adjust=False).mean()
    ema200 = close.ewm(span=200, adjust=False).mean()

    r = rsi(close, 14)
    mf = mfi(df, 14)

    macd = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
    macd_sig = macd.ewm(span=9, adjust=False).mean()

    vol_avg = volume.rolling(20).mean()

    bb_mid = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_low = bb_mid - 2 * bb_std
    bb_up = bb_mid + 2 * bb_std

    range_high = high.rolling(100).max()
    range_low = low.rolling(100).min()
    orta = (range_high + range_low) / 2

    onceki_dip = low.rolling(20).min().shift(1)

    if pd.isna(r.iloc[-1]) or pd.isna(mf.iloc[-1]) or pd.isna(vol_avg.iloc[-1]):
        return "YETERSİZ", 0

    c = float(close.iloc[-1])
    o = float(open_.iloc[-1])
    v = float(volume.iloc[-1])

    dip_supurme = low.iloc[-1] < onceki_dip.iloc[-1] and c > onceki_dip.iloc[-1]
    dip_bolge = c < orta.iloc[-1]
    bb_tepki = low.iloc[-1] <= bb_low.iloc[-1] and c > bb_low.iloc[-1]
    rsi_donus = r.iloc[-1] < 42 and r.iloc[-1] > r.iloc[-2]
    mfi_donus = mf.iloc[-1] < 45 and mf.iloc[-1] > mf.iloc[-2]
    hacim_tepki = v > vol_avg.iloc[-1] and c > o

    dip_score = 0
    dip_score += 2 if dip_supurme else 0
    dip_score += 1 if dip_bolge else 0
    dip_score += 1 if bb_tepki else 0
    dip_score += 1 if rsi_donus else 0
    dip_score += 1 if mfi_donus else 0
    dip_score += 1 if hacim_tepki else 0

    dipten_al = dip_score >= 3

    kurum_topluyor = c > o and v > vol_avg.iloc[-1] * 1.5 and mf.iloc[-1] > mf.iloc[-2]
    kurum_satiyor = c < o and v > vol_avg.iloc[-1] * 1.5 and mf.iloc[-1] < mf.iloc[-2]

    guvenli_al = (
        c > ema20.iloc[-1] and
        ema20.iloc[-1] > ema50.iloc[-1] and
        45 < r.iloc[-1] < 68 and
        macd.iloc[-1] > macd_sig.iloc[-1] and
        v > vol_avg.iloc[-1] and
        not kurum_satiyor
    )

    al = (
        c > ema20.iloc[-1] and
        r.iloc[-1] > 50 and
        macd.iloc[-1] > macd_sig.iloc[-1]
    )

    son_yukselis = (c - low.rolling(20).min().iloc[-1]) / max(low.rolling(20).min().iloc[-1], 0.001) * 100

    gec_kaldin = son_yukselis > 25 and r.iloc[-1] > 70

    kar_al = (
        son_yukselis > 15 and
        r.iloc[-1] > 62 and
        c < close.iloc[-2] and
        macd.iloc[-1] < macd_sig.iloc[-1]
    )

    sat = (
        c < ema50.iloc[-1] and
        r.iloc[-1] < 45 and
        macd.iloc[-1] < macd_sig.iloc[-1]
    )

    puan = 0
    puan += 25 if dipten_al else 0
    puan += 20 if kurum_topluyor else 0
    puan += 15 if guvenli_al else 0
    puan += 5 if v > vol_avg.iloc[-1] * 2 else 0
    puan += 5 if 50 < r.iloc[-1] < 70 else 0
    puan += 5 if c > ema20.iloc[-1] else 0
    puan += 5 if ema20.iloc[-1] > ema50.iloc[-1] else 0
    puan += 5 if c > ema200.iloc[-1] else 0
    puan += 5 if macd.iloc[-1] > macd_sig.iloc[-1] else 0

    if kurum_satiyor:
        sinyal = "KURUM SATIYOR"
    elif sat:
        sinyal = "SAT"
    elif kar_al:
        sinyal = "KÂR AL"
    elif gec_kaldin:
        sinyal = "GEÇ KALDIN"
    elif dipten_al:
        sinyal = "DİPTEN AL"
    elif kurum_topluyor:
        sinyal = "KURUM TOPLUYOR"
    elif guvenli_al:
        sinyal = "GÜVENLİ AL"
    elif al:
        sinyal = "AL"
    else:
        sinyal = "BEKLE"

    return sinyal, int(puan)

# =========================
# KARAR
# =========================
def karar(p):
    if p >= 90:
        return "ELMAS"
    if p >= 80:
        return "ÇOK GÜÇLÜ"
    if p >= 70:
        return "AL ADAYI"
    if p >= 60:
        return "TAKİP"
    return "ZAYIF"

# =========================
# TARAMA
# =========================
rows = []

for i, sym in enumerate(symbols, 1):
    print(f"{i}/{len(symbols)} taranıyor: {sym}")

    try:
        df = download_data(sym)

        if df is None:
            print(sym, "veri yok veya yetersiz")
            continue

        daily = df.copy()
        weekly = resample_ohlcv(df, "W")
        monthly = resample_ohlcv(df, "M")

        d_sig, d_puan = signal_engine(daily)
        w_sig, w_puan = signal_engine(weekly)
        m_sig, m_puan = signal_engine(monthly)

        if m_sig == "YETERSİZ":
            toplam_puan = int(d_puan * 0.60 + w_puan * 0.40)
        elif w_sig == "YETERSİZ":
            toplam_puan = int(d_puan)
        else:
            toplam_puan = int(d_puan * 0.50 + w_puan * 0.30 + m_puan * 0.20)
        rows.append({
            "Hisse": sym,
            "Günlük": d_sig,
            "Haftalık": w_sig,
            "Aylık": m_sig,
            "Günlük Puan": d_puan,
            "Haftalık Puan": w_puan,
            "Aylık Puan": m_puan,
            "Toplam Puan": toplam_puan,
            "Karar": karar(toplam_puan)
        })

    except Exception as e:
        print(sym, "hata:", e)

result = pd.DataFrame(rows)

if result.empty:
    print("Sonuç oluşmadı. Veri kaynağı çalışmamış olabilir.")
else:
    result = result.sort_values("Toplam Puan", ascending=False)

    file_name = f"bist_tum_tarama_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

    adaylar = result[
        result["Günlük"].isin(["DİPTEN AL", "GÜVENLİ AL", "KURUM TOPLUYOR", "AL"])
    ]

    with pd.ExcelWriter(file_name, engine="openpyxl") as writer:
        result.to_excel(writer, index=False, sheet_name="Tüm Sonuçlar")
        result.head(30).to_excel(writer, index=False, sheet_name="En Güçlü 30")
        adaylar.to_excel(writer, index=False, sheet_name="Bugün Adaylar")

    print("Tarama bitti.")
    print("Dosya hazır:", file_name)

    # files.download(file_name)

    import requests

TELEGRAM_TOKEN = "8819448065:AAHs_FNHi4zwlLhiUHExKAs7EOco-CaEl0g"
CHAT_ID = "5920392803"

def telegram_gonder(mesaj):
    url = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}/sendMessage"
    data = {
        "chat_id": CHAT_ID,
        "text": mesaj,
        "parse_mode": "HTML"
    }
    requests.post(url, data=data)

top10 = result.head(10)

mesaj = "🚀 <b>AYDIN BIST ROBOTU - BUGÜNÜN EN GÜÇLÜ 10 HİSSESİ</b>\n\n"

for i, row in top10.iterrows():
    mesaj += f"📌 <b>{row['Hisse']}</b>\n"
    mesaj += f"Günlük: {row['Günlük']}\n"
    mesaj += f"Haftalık: {row['Haftalık']}\n"
    mesaj += f"Aylık: {row['Aylık']}\n"
    mesaj += f"Puan: {row['Toplam Puan']}\n"
    mesaj += f"Karar: {row['Karar']}\n\n"

#telegram_gonder(mesaj)

#print("Telegram mesajı gönderildi.")
if not result.empty:
    display(result.head(30))

Taranacak hisse sayısı: 245
1/245 taranıyor: AEFES


/tmp/ipykernel_6626/3035213840.py:100: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_6626/3035213840.py:101: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_6626/3035213840.py:102: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_6626/3035213840.py:103: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_6626/3035213840.py:104: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()


2/245 taranıyor: AGHOL


/tmp/ipykernel_6626/3035213840.py:100: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_6626/3035213840.py:101: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_6626/3035213840.py:102: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_6626/3035213840.py:103: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_6626/3035213840.py:104: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()


3/245 taranıyor: AKBNK


...

/tmp/ipykernel_6626/3035213840.py:100: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_6626/3035213840.py:101: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_6626/3035213840.py:102: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_6626/3035213840.py:103: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_6626/3035213840.py:104: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()
/tmp/ipykernel_6626/3035213840.py:100: FutureWarning: 'M' is deprecated and will be remov

244/245 taranıyor: ZEDUR
245/245 taranıyor: ZOREN


/tmp/ipykernel_6626/3035213840.py:100: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Open": df["Open"].resample(rule).first(),
/tmp/ipykernel_6626/3035213840.py:101: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "High": df["High"].resample(rule).max(),
/tmp/ipykernel_6626/3035213840.py:102: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Low": df["Low"].resample(rule).min(),
/tmp/ipykernel_6626/3035213840.py:103: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Close": df["Close"].resample(rule).last(),
/tmp/ipykernel_6626/3035213840.py:104: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  "Volume": df["Volume"].resample(rule).sum()


Tarama bitti.
Dosya hazır: bist_tum_tarama_20260606_1803.xlsx


,Hisse,Günlük,Haftalık,Aylık,Günlük Puan,Haftalık Puan,Aylık Puan,Toplam Puan,Karar
192,SOKE,SAT,GÜVENLİ AL,YETERSİZ,55,40,0,49,ZAYIF
142,MAALT,KURUM TOPLUYOR,KURUM SATIYOR,YETERSİZ,60,25,0,46,ZAYIF
83,GEDIK,GÜVENLİ AL,KURUM TOPLUYOR,YETERSİZ,35,60,0,45,ZAYIF
145,MAVI,KURUM TOPLUYOR,BEKLE,YETERSİZ,65,15,0,45,ZAYIF
119,KARTN,GÜVENLİ AL,KURUM TOPLUYOR,YETERSİZ,40,45,0,42,ZAYIF
182,SAYAS,GÜVENLİ AL,GEÇ KALDIN,YETERSİZ,40,45,0,42,ZAYIF
204,TLMAN,AL,KURUM TOPLUYOR,YETERSİZ,25,65,0,41,ZAYIF
84,GEDZA,AL,KURUM TOPLUYOR,YETERSİZ,25,65,0,41,ZAYIF
37,BRSAN,KURUM TOPLUYOR,KURUM TOPLUYOR,YETERSİZ,40,40,0,40,ZAYIF
123,KFEIN,GÜVENLİ AL,GEÇ KALDIN,YETERSİZ,40,40,0,40,ZAYIF
